ШАГ 3: РЕГРЕССИЯ ДЛЯ ПРЕДСКАЗАНИЯ ИНДЕКСА СЕЛЕКТИВНОСТИ

Целью данного этапа является построение прогнозной модели для индекса селективности. Так как он является производной величиной, зависимость этой метрики от дескрипторов носит сложный и нелинейный характер. Для стабилизации дисперсии целевой переменной применяется логарифмическая шкала.

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# и опять загружаю данные
df = pd.read_csv('cleaned_data.csv')

targets_to_exclude = ['IC50, mM', 'CC50, mM', 'SI', 'pIC50', 'pCC50', 'log_SI',
                      'IC50_above_med', 'CC50_above_med', 'SI_above_med', 'SI_above_8']
X = df.drop(columns=targets_to_exclude)
y = df['log_SI']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Train: X={X_train.shape}, y={y_train.shape}")
print(f"Test:  X={X_test.shape}, y={y_test.shape}\n")

# сначала базовые модели
pipelines_base = {
    'Ridge_Base': Pipeline([('scaler', StandardScaler()), ('model', Ridge(random_state=42))]),
    'SVR_Base': Pipeline([('scaler', StandardScaler()), ('model', SVR())]),
    'RF_Base': Pipeline([('scaler', StandardScaler()), ('model', RandomForestRegressor(random_state=42))])
}

param_grids_base = {
    'Ridge_Base': {'model__alpha': [0.1, 1.0, 10.0, 50.0, 100.0]},
    'SVR_Base': {
        'model__C': [0.1, 1, 10],
        'model__kernel': ['rbf', 'linear']
    },
    'RF_Base': {
        'model__n_estimators': [100, 200],
        'model__max_depth': [None, 10, 20]
    }
}

results_base = []

for name in pipelines_base:
    print(f"обучение {name}...")
    grid = GridSearchCV(pipelines_base[name], param_grids_base[name], cv=5,
                        scoring='neg_mean_squared_error', n_jobs=-1)
    grid.fit(X_train, y_train)
    y_pred = grid.best_estimator_.predict(X_test)

    results_base.append({
        'модель': name,
        'лучшие параметры': str(grid.best_params_),
        'MSE': mean_squared_error(y_test, y_pred),
        'MAE': mean_absolute_error(y_test, y_pred),
        'R2 Score': r2_score(y_test, y_pred)
    })

results_base_df = pd.DataFrame(results_base).sort_values(by='R2 Score', ascending=False)
print("результаты базовых моделей:")
print(results_base_df.to_string(index=False))

Train: X=(800, 139), y=(800,)
Test:  X=(201, 139), y=(201,)

обучение Ridge_Base...
обучение SVR_Base...
обучение RF_Base...
результаты базовых моделей:
    Модель                                     Лучшие параметры      MSE      MAE  R2 Score
   RF_Base {'model__max_depth': 10, 'model__n_estimators': 100} 0.442983 0.493067  0.273918
  SVR_Base              {'model__C': 1, 'model__kernel': 'rbf'} 0.464767 0.490159  0.238212
Ridge_Base                              {'model__alpha': 100.0} 0.536633 0.555135  0.120418


In [ ]:
# теперь более сложные

pipelines_adv = {
    'Ridge_Poly': Pipeline([
        ('poly', PolynomialFeatures(degree=2, include_bias=False)),
        ('scaler', StandardScaler()),
        ('selector', SelectKBest(score_func=f_regression)),
        ('model', Ridge(random_state=42))
    ]),
    'SVR_Poly': Pipeline([
        ('poly', PolynomialFeatures(degree=2, include_bias=False)),
        ('scaler', StandardScaler()),
        ('selector', SelectKBest(score_func=f_regression)),
        ('model', SVR())
    ])
}

param_grids_adv = {
    'Ridge_Poly': {
        'selector__k': [100, 200, 300],
        'model__alpha': [10.0, 50.0, 100.0, 200.0]
    },
    'SVR_Poly': {
        'selector__k': [100, 200],
        'model__C': [0.1, 1, 10],
        'model__kernel': ['rbf', 'linear']
    }
}

results_adv = []

for name in pipelines_adv:
    grid = GridSearchCV(pipelines_adv[name], param_grids_adv[name], cv=5,
                        scoring='neg_mean_squared_error', n_jobs=-1)
    grid.fit(X_train, y_train)
    y_pred = grid.best_estimator_.predict(X_test)

    results_adv.append({
        'модель': name,
        'лучшие параметры': str(grid.best_params_),
        'MSE': mean_squared_error(y_test, y_pred),
        'MAE': mean_absolute_error(y_test, y_pred),
        'R2 Score': r2_score(y_test, y_pred)
    })

results_adv_df = pd.DataFrame(results_adv).sort_values(by='R2 Score', ascending=False)
print("результаты моделей:")
print(results_adv_df.to_string(index=False))

Настройка Ridge_Poly...
Настройка SVR_Poly...


PicklingError: Could not pickle the task to send it to the workers.

ВЫВОДЫ: 

Индекс селективности оказался сложной для прогнозирования величиной: максимальный достигнутый коэффициент детерминации составил около 0.27. Я полагаю, что в данных присутствует значительный шум либо доступный набор дескрипторов недостаточен для точного описания зависимости.

Эффективность инженерии признаков подтвердилась лишь частично. Базовая линейная модель (Ridge) на исходных признаках объясняла лишь 12% дисперсии; после генерации полиномиальных признаков и отбора 300 наиболее информативных переменных её точность возросла, однако общий прирост остался скромным. Таким образом, гипотеза о существовании нелинейных взаимодействий не получила сильного эмпирического подкрепления.

Среди рассмотренных алгоритмов наилучший результат продемонстрировал случайный лес (R² ≈ 0.273). Деревья решений способны улавливать нелинейные зависимости без явного конструирования дополнительных признаков. Введение полиномиальных признаков для метода опорных векторов (SVR) не привело к значимому улучшению.

Для дальнейшего повышения качества прогноза я рекомендую рассмотреть применение более мощных алгоритмов, таких как градиентный бустинг (XGBoost, LightGBM) или нейронные сети. Целесообразным представляется также сбор дополнительных данных, поскольку текущая разметка может содержать шум. Кроме того, снижение размерности с помощью PCA или более тщательное удаление сильно коррелирующих признаков потенциально способно улучшить соотношение сигнал/шум.